In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local")\
        .appName("Colab")\
        .config('spark.ui.port', '4050')\
        .getOrCreate()

spark

In [3]:
df = spark.read.format("csv").load("housing.csv", header=True, inferSchema=True)

df.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



In [6]:
display(df.show())

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

None

In [7]:
from pyspark.sql.functions import monotonically_increasing_id

df = df.withColumn('id', monotonically_increasing_id())

df = df[['id'] + df.columns[:-1]]

df.show(3)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  0|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  1|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
only s

In [22]:
from pyspark.sql.functions import col
df.select(col("latitude")).show()

+--------+
|latitude|
+--------+
|   37.88|
|   37.86|
|   37.85|
|   37.85|
|   37.85|
|   37.85|
|   37.84|
|   37.84|
|   37.84|
|   37.84|
|   37.85|
|   37.85|
|   37.85|
|   37.84|
|   37.85|
|   37.85|
|   37.85|
|   37.85|
|   37.84|
|   37.84|
+--------+
only showing top 20 rows



In [24]:
quantiles = df.approxQuantile("latitude", [0.25, 0.75], 0.01)
print(quantiles)
Q1, Q3 = quantiles[0], quantiles[1]
IQR = Q3 - Q1
print(IQR)

[33.93, 37.69]
3.759999999999998


In [25]:
lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
lower_bound, upper_bound

(28.290000000000003, 43.33)

In [30]:
outliers_df = df.filter((col("latitude") < lower_bound) | (col("latitude") > upper_bound))
outliers_df.show()

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+



# Remove outliers

In [31]:
numericals = ["longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms", "population", "households", "median_income", "median_house_value"]
for feature in numericals:
  quantiles = df.approxQuantile(feature, [0.25, 0.75], 0.01)
  Q1, Q3 = quantiles[0], quantiles[1]
  IQR = Q3 - Q1
  lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
  df = df.filter((col(feature) > lower_bound) & (col(feature) < upper_bound))

df.show(5)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  3|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  4|  -122.25|   37.85|              52.0|     1627.0|         280.0|     565.0|     259.0|       3.8462|          342200.0|       NEAR BAY|
|  5|  -122.25|   37.85|              52.0|      919.0|         213.0|     413.0|     193.0|       4.0368|          269700.0|       NEAR BAY|
|  6| 

In [33]:
df.count()

16534

In [35]:
df.groupBy('ocean_proximity').agg({column: "avg" for column in df.columns[1:-1]}).show()

+---------------+------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+-----------------------+-----------------------+
|ocean_proximity|   avg(households)|     avg(latitude)|   avg(population)|avg(total_bedrooms)|     avg(longitude)|avg(median_income)|  avg(total_rooms)|avg(median_house_value)|avg(housing_median_age)|
+---------------+------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+-----------------------+-----------------------+
|         ISLAND| 297.3333333333333| 33.36666666666667| 751.6666666666666|              439.0|-118.37666666666667|2.9427000000000003|            1734.0|      334066.6666666667|     44.333333333333336|
|     NEAR OCEAN| 424.1963249516441|34.709550290135496| 1128.901837524178| 453.41392649903287| -119.2904642166345|3.5554962282398423|2102.6537717601545|     217431.14119922632|     30.176982591876

In [39]:
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf

def interact(x, y):
  return x*y

interact_udf = udf(interact, FloatType())
df = df.withColumn("latitude*income", interact_udf('latitude', 'median_income'))
df.show(5)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|latitude*income|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|       274.6926|
|  3|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|      213.59134|
|  4|  -122.25|   37.85|              52.0|     1627.0|         280.0|     565.0|     259.0|       3.8462|          342200.0|       NEAR BAY|      145.57867|
|  5|  -122.25|   37.85|              52.0|      919

In [40]:
train, test = df.randomSplit([0.7, 0.3])

In [55]:
numerical_features = train.columns
numerical_features.remove("ocean_proximity")
numerical_features.remove("median_house_value")
numerical_features.remove("id")

In [56]:
from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols=numerical_features, outputCols=numerical_features)

imputer = imputer.fit(train)

train = imputer.transform(train)
test = imputer.transform(test)

train.show(3)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|latitude*income|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|       274.6926|
|  3|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|      213.59134|
|  4|  -122.25|   37.85|              52.0|     1627.0|         280.0|     565.0|     259.0|       3.8462|          342200.0|       NEAR BAY|      145.57867|
+---+---------+--------+------------------+---------

In [57]:
from pyspark.ml.feature import VectorAssembler

numerical_vector_assembler = VectorAssembler(inputCols=numerical_features, outputCol='numerical_features_vector')

train = numerical_vector_assembler.transform(train)
test = numerical_vector_assembler.transform(test)

train.show(2)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|latitude*income|numerical_features_vector|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|       274.6926|     [-122.24,37.85,52...|
|  3|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|      213.59134|     [-122.25,37.85,52...|
+---+---------+--------+------------------+-----------+--------------+----------

In [59]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol='numerical_features_vector',
                        outputCol='numerical_features_vector_scaled',
                        withStd=True,
                        withMean=True)

scaler = scaler.fit(train)

train = scaler.transform(train)
test = scaler.transform(test)

train.show(3)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|latitude*income|numerical_features_vector|numerical_features_vector_scaled|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|       274.6926|     [-122.24,37.85,52...|            [-1.3098824588784...|
|  3|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       

In [60]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol='ocean_proximity',
                        outputCol='ocean_category_index')

indexer = indexer.fit(train)
train = indexer.transform(train)
test = indexer.transform(test)

train.show(3)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+--------------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|latitude*income|numerical_features_vector|numerical_features_vector_scaled|ocean_category_index|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+--------------------+
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|       274.6926|     [-122.24,37.85,52...|            [-1.3098824588784...|                 3.0|
|  3|  -122.25|   37.85|              52.0|     

In [61]:
from pyspark.ml.feature import OneHotEncoder

one_hot_encoder = OneHotEncoder(inputCol='ocean_category_index',
                                outputCol='ocean_category_one_hot')

one_hot_encoder = one_hot_encoder.fit(train)

train = one_hot_encoder.transform(train)
test = one_hot_encoder.transform(test)

train.show(3)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+--------------------+----------------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|latitude*income|numerical_features_vector|numerical_features_vector_scaled|ocean_category_index|ocean_category_one_hot|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+--------------------+----------------------+
|  2|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|       274.6926|     [-122.24,37.85,52...|            [-1.3098824588784...| 

In [63]:
assembler = VectorAssembler(inputCols=['numerical_features_vector_scaled',
                                       'ocean_category_one_hot'],
                            outputCol='final_feature_vector')

train = assembler.transform(train)
test = assembler.transform(test)

In [64]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol='final_feature_vector',
                      labelCol='median_house_value')

lr = lr.fit(train)

In [65]:
pred_test_df = lr.transform(test).withColumnRenamed('prediction', 'predicted_median_house_value')

pred_test_df.show(5)

+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+--------------------+----------------------+--------------------+----------------------------+
| id|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|latitude*income|numerical_features_vector|numerical_features_vector_scaled|ocean_category_index|ocean_category_one_hot|final_feature_vector|predicted_median_house_value|
+---+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------+-------------------------+--------------------------------+--------------------+----------------------+--------------------+----------------------------+
|  8|  -122.26|   37.84|              42.0|     2555.0|         665

In [67]:
evaluator = pred_test_df.select(col('predicted_median_house_value'), col('median_house_value')).rdd

evaluator.take(3)

[Row(predicted_median_house_value=200043.974472681, median_house_value=226700.0),
 Row(predicted_median_house_value=251948.1107100304, median_house_value=241800.0),
 Row(predicted_median_house_value=195217.25498002567, median_house_value=159200.0)]

In [68]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="median_house_value", predictionCol="predicted_median_house_value", metricName="rmse")
evaluator_mse = RegressionEvaluator(labelCol="median_house_value", predictionCol="predicted_median_house_value", metricName="mse")
evaluator_mae = RegressionEvaluator(labelCol="median_house_value", predictionCol="predicted_median_house_value", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="median_house_value", predictionCol="predicted_median_house_value", metricName="r2")

rmse = evaluator_rmse.evaluate(pred_test_df)
mse = evaluator_mse.evaluate(pred_test_df)
mae = evaluator_mae.evaluate(pred_test_df)
r2 = evaluator_r2.evaluate(pred_test_df)


print(f"RMSE: {rmse}")
print(f"MSE: {mse}")
print(f"MAE: {mae}")
print(f"R2: {r2}")

RMSE: 54722.403553607524
MSE: 2994541450.6838775
MAE: 41229.87379430336
R2: 0.6401895570512016
